In [ ]:
# |default_exp utils.devices
# |export
from typing import Literal
import numpy as np
import torch


In [ ]:
# |export
def get_torch_device(
    prefer: Literal["auto", "cuda", "mps", "cpu"] = "auto",
) -> torch.device:
    """Resolve the Torch device used by notebooks and examples."""
    assert prefer in ["auto", "cuda", "mps", "cpu"], (
        f"prefer must be one of auto, cuda, mps, or cpu, got {prefer!r}"
    )
    if prefer == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    if prefer == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA was requested, but torch.cuda.is_available() is False")
        return torch.device("cuda")
    if prefer == "mps":
        if not torch.backends.mps.is_available():
            raise RuntimeError("MPS was requested, but torch.backends.mps.is_available() is False")
        return torch.device("mps")
    return torch.device("cpu")


def linalg_work_device(device: torch.device) -> torch.device:
    """Return the device to use for linalg calls with known MPS gaps."""
    device = torch.device(device)
    if device.type == "mps":
        return torch.device("cpu")
    return device


def as_numpy(x: torch.Tensor | np.ndarray) -> np.ndarray:
    """Convert a tensor or NumPy array to a CPU NumPy array for plotting/scipy/sklearn."""
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    if isinstance(x, np.ndarray):
        return x
    return np.asarray(x)


def torch_generator_for(
    device: torch.device, seed: int | None = None
) -> torch.Generator | None:
    """Create a Torch random generator on devices that support explicit generators."""
    device = torch.device(device)
    if device.type == "mps":
        return None
    generator = torch.Generator(device=device)
    if seed is not None:
        generator.manual_seed(seed)
    return generator
